**Objective**:

The Spark Declarative Pipeline is designed to ingest chunk2.csv from localized Volume storage into the Bronze Layer. It utilizes a metadata-driven approach to ensure schema consistency, idempotency, and high-precision audit tracking.

In [0]:
import json
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# --- 1. CONFIGURATION ---
dbutils.widgets.text("datasets_json", '[]', "Datasets List")
try:
    from schema_config import SOURCE_BASE_PATH, DEST_CATALOG, DEST_SCHEMA
except:
    SOURCE_BASE_PATH = "/Volumes/data_landing/data_raw"
    DEST_CATALOG     = "data_bronze"
    DEST_SCHEMA      = "bronze"

def spark_declarative_engine(dataset_name):
    """
    Modular engine to process chunk2.csv.
    - Skips if hierarchy is missing.
    - Ensures idempotency by checking 'source' column in Delta.
    """
    clean_name = dataset_name.lower()
    chunks_dir = f"{SOURCE_BASE_PATH}/{clean_name}/chunks"
    source_path = f"{chunks_dir}/chunk2.csv"
    target_table = f"{DEST_CATALOG}.{DEST_SCHEMA}.{clean_name}"

    # --- 1. PATH VALIDATION ---
    try:
        # Check if the 'chunks' folder exists
        dbutils.fs.ls(chunks_dir)
        # Check if 'chunk2.csv' exists specifically
        files = [f.name for f in dbutils.fs.ls(chunks_dir)]
        if "chunk2.csv" not in files:
            print(f"[SKIP] {dataset_name}: 'chunks' folder exists but 'chunk2.csv' missing.")
            return
    except Exception as e:
        if "java.io.FileNotFoundException" in str(e):
            print(f"[SKIP] {dataset_name}: 'chunks' folder hierarchy not found.")
        else:
            print(f"[ERROR] {dataset_name}: Access issue: {str(e)[:50]}")
        return

    # --- 2. IDEMPOTENCY CHECK ---
    # Check if this specific table already contains data from 'chunk2.csv'
    try:
        if spark.catalog.tableExists(target_table):
            already_ingested = spark.table(target_table) \
                .filter(F.col("source") == "chunk2.csv") \
                .limit(1).count() > 0
            
            if already_ingested:
                print(f"[IDEMPOTENT] {dataset_name}: chunk2.csv already ingested. Skipping.")
                return
    except Exception as e:
        print(f"[WARNING] {dataset_name}: Could not verify idempotency, proceeding. Error: {str(e)[:50]}")

    print(f"[INGESTING] Processing {dataset_name}...")

    # --- 3. DECLARATIVE READ (All-String) ---
    df_raw = (spark.read
              .option("header", "true")
              .option("inferSchema", "false") # Force String
              .csv(source_path)
              .select("*", "_metadata.file_path"))

    # Identify data columns (excluding metadata we will generate)
    data_cols = [c for c in df_raw.columns if c not in ["file_path", "load_dt", "source"]]
    
    df_final = df_raw.select(
        *[F.col(c).cast(StringType()) for c in data_cols],
        F.current_timestamp().alias("load_dt"),
        F.element_at(F.split(F.col("file_path"), "/"), -1).alias("source")
    )

    # --- 4. WRITE TO DELTA ---
    (df_final.write
     .format("delta")
     .mode("append")
     .option("mergeSchema", "true")
     .saveAsTable(target_table))

    print(f"[SUCCESS] {dataset_name}: chunk2.csv integrated into {target_table}")

# --- 2. ORCHESTRATION ---
if __name__ == "__main__":
    raw_input = dbutils.widgets.get("datasets_json")
    try:
        datasets = json.loads(raw_input)
        if datasets:
            print(f"SDP Batch started for {len(datasets)} dataset(s).")
            print("-" * 50)
            for ds in datasets:
                spark_declarative_engine(ds)
            print("-" * 50)
        else:
            print("No datasets provided in widget.")
    except Exception as e:
        print(f"Pipeline Failed: {str(e)}")


**Validation Objectives**

Run this suite to confirm that the SDP engine successfully processed the data while adhering to the Bronze Layer governance standards.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, TimestampType
import json

def test_sdp_ingestion(dataset_name):
    """
    Modular test runner to validate the Spark Declarative Pipeline output.
    Returns a dictionary for reporting.
    """
    table_name = f"{DEST_CATALOG}.{DEST_SCHEMA}.{dataset_name.lower()}"
    test_report = {
        "Dataset": dataset_name,
        "Table": table_name,
        "Status": "SKIPPED",
        "Records_Found": 0,
        "Failures": []
    }

    # 1. Check Table Existence
    if not spark.catalog.tableExists(table_name):
        test_report["Failures"].append("Table missing in catalog.")
        return test_report

    try:
        # 2. Verify Presence of chunk2.csv
        # This confirms that the SDP logic actually executed for this dataset
        df = spark.table(table_name).filter(F.col("source") == "chunk2.csv")
        row_count = df.count()
        test_report["Records_Found"] = row_count

        if row_count == 0:
            test_report["Status"] = "NOT_PROCESSED"
            test_report["Failures"].append("No records found with source 'chunk2.csv'.")
            return test_report

        # 3. Schema Validation (All-String Bronze Standard)
        # Verify all columns EXCEPT load_dt are StringType
        invalid_types = []
        for field in df.schema.fields:
            if field.name == "load_dt":
                if not isinstance(field.dataType, TimestampType):
                    invalid_types.append(f"load_dt is {field.dataType}, expected TimestampType")
            else:
                if not isinstance(field.dataType, StringType):
                    invalid_types.append(f"{field.name} is {field.dataType}, expected StringType")
        
        if invalid_types:
            test_report["Failures"].extend(invalid_types)
            test_report["Status"] = "SCHEMA_FAIL"
        else:
            test_report["Status"] = "PASSED"

    except Exception as e:
        test_report["Status"] = "ERROR"
        test_report["Failures"].append(str(e)[:100])

    return test_report

# --- ORCHESTRATION & VISUAL REPORT ---
if __name__ == "__main__":
    raw_input = dbutils.widgets.get("datasets_json")
    datasets = json.loads(raw_input)
    
    results = []
    for ds in datasets:
        results.append(test_sdp_ingestion(ds))

    # Convert to DataFrame for a clean visual report in Databricks
    summary_df = spark.createDataFrame(results)
    
    print("--- SPARK DECLARATIVE PIPELINE (SDP) VALIDATION SUMMARY ---")
    display(summary_df.select("Status", "Dataset", "Records_Found", "Failures"))